In [50]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [53]:
df=pd.read_csv('qoute_dataset.csv')

In [54]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   3038 non-null   object
 1   Author  3038 non-null   object
dtypes: object(2)
memory usage: 47.6+ KB


In [56]:
df.shape

(3038, 2)

In [57]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [58]:
quotes = quotes.str.lower()

In [59]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [60]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: object

In [97]:
from tensorflow.keras.preprocessing.text import  Tokenizer

In [98]:
vocab_size=8978
tokenizer=Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [63]:
quotes

0       “the world as we have created it is a process ...
1       “it is our choices harry that show what we tru...
2       “there are only two ways to live your life one...
3       “the person be it gentleman or lady who has no...
4       “imperfection is beauty madness is genius and ...
                              ...                        
3033         the past beats inside me like a second heart
3034    damn claire warn a guy before you do a facepla...
3035    can you be a girl for a few secondsim always a...
3036    thats what fiction is for its for getting at t...
3037    if we have no peace it is because we have forg...
Name: quote, Length: 3038, dtype: object

In [64]:
word_index=tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [65]:
sequence = tokenizer.texts_to_sequences(quotes)

In [66]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [67]:
for i in range(3):
    print(sequence[0])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]


In [68]:
X=[]
y=[]

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq=seq[:i]
        output_seq=seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [ ]:
X
# run at your own risk

In [70]:
len(X)

85270

In [71]:
len(y)

85270

In [72]:
max_len=max(len(x) for x in X)
print(max_len)

745


In [73]:
from tensorflow.keras.preprocessing.sequence import  pad_sequences
X_padded = pad_sequences(X,maxlen=max_len,padding='pre')

In [74]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85270, 745), dtype=int32)

In [75]:
X[0]

[713]

In [76]:
y=np.array(y)

In [77]:
y

array([ 62,  29,  19, ...,   3, 169, 101], shape=(85270,))

In [78]:
X_padded.shape

(85270, 745)

In [79]:
y.shape

(85270,)

In [80]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [81]:
y_one_hot.shape

(85270, 8978)

In [82]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [83]:
embedding_dim = 50
rnn_units = 128

In [91]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

In [92]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [93]:
rnn_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [94]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [95]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [96]:
lstm_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [33]:
epochs=10
batch_size=128

In [40]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 45s 69ms/step - accuracy: 0.0430 - loss: 6.7374 - val_accuracy: 0.0536 - val_loss: 6.5957
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.0711 - loss: 6.1619 - val_accuracy: 0.0843 - val_loss: 6.3409
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.0964 - loss: 5.8197 - val_accuracy: 0.0963 - val_loss: 6.2710
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1153 - loss: 5.5194 - val_accuracy: 0.1071 - val_loss: 6.2709
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1293 - loss: 5.2574 - val_accuracy: 0.1114 - val_loss: 6.3126
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1428 - loss: 5.0207 - val_accuracy: 0.1116 - val_loss: 6.3517
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1564 - loss: 4.8016 - val_accuracy: 0.1091 - val_loss: 6.4229
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1722 - loss: 4.5952 - 

7 min taken to train
on gpu T4

In [41]:
#rnn_model.save("rnn_model.keras")

In [34]:
epochs =100
batch_size=128

history_lstm = lstm_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 54ms/step - accuracy: 0.0401 - loss: 6.7474 - val_accuracy: 0.0496 - val_loss: 6.6680
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 53ms/step - accuracy: 0.0580 - loss: 6.3206 - val_accuracy: 0.0634 - val_loss: 6.5561
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.0787 - loss: 6.0559 - val_accuracy: 0.0873 - val_loss: 6.4485
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0982 - loss: 5.8276 - val_accuracy: 0.0946 - val_loss: 6.4143
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1097 - loss: 5.6373 - val_accuracy: 0.1002 - val_loss: 6.4106
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1203 - loss: 5.4607 - val_accuracy: 0.1027 - val_loss: 6.4100
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1296 - loss: 5.2979 - val_accuracy: 0.1055 - val_loss: 6.4467
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.1367 - loss: 5

In [35]:
lstm_model.save("lstm_model.h5")

lstm time taken gpu T4 54 mins

In [85]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")

In [86]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [87]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [118]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [119]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

longingit


In [124]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [125]:
seed = "are you a fool "
generate_text = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generate_text)

are you a fool  وجبة sequestered regrets againi againi rescued influence everyone هي covers


In [126]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [127]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)
     